# Clase 2 · PL/SQL y optimización de consultas

**Caso:** Mercado Sur, una organización ficticia que vende por web y en tienda. **Pregunta de la clase:** ¿cómo convertimos una consulta útil en una pieza reutilizable y eficiente, sin perder trazabilidad ni cambiar su significado?

Este archivo es **autocontenido**: genera sus propios datos y, si falta alguna librería, la instala solo. Se corre de arriba hacia abajo con `Shift` + `Enter`.

**Cómo trabajamos.** Cada paso tiene tres momentos: qué vamos a hacer, la celda y qué vimos. Donde veas **PREDICCIÓN**, respondé *antes* de ejecutar: son dos celdas, la primera con la pregunta y `respuesta = None` (cambiá `None` por tu letra, por ejemplo `"B"`) y la segunda, que comprueba.

> **Nivel de realización.** El PL/SQL de Oracle que se muestra está **especificado, no ejecutado en este entorno**. Lo que ejecutamos son **equivalentes con DuckDB**: una *macro* (una función escrita en SQL) hace de función, y los planes de ejecución que leemos son los de DuckDB, no los de Oracle. El criterio de lectura es el mismo.

| Paso | Qué hacemos |
| --- | --- |
| 1 | Armamos los datos de Mercado Sur |
| 2 | El problema: tres reportes, tres números |
| 3 | Una base de PL/SQL, en simple |
| 4 | Los siete pasos para desarrollar una función |
| 5 | Tu turno: una segunda función con casos borde |
| 6 | Un error que compila y miente |
| 7 | Optimizar una consulta con evidencia |
| 8 | La plantilla de evidencia de seis campos |

## Preparación

In [ ]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas", "duckdb", "matplotlib")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

In [ ]:
import time
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display

con = duckdb.connect()          # el motor de SQL, dentro del notebook

def _sql(line, cell):
    """Las celdas que empiezan con %%sql ejecutan SQL. El resultado se guarda en la variable que nombres (por ejemplo: %%sql mi_tabla)."""
    nombre = line.strip().split()[0] if line.strip() else "resultado"
    try:
        df = con.sql(cell).df()
    except Exception as e:
        print("Error de SQL:", e)
        return
    get_ipython().user_ns[nombre] = df
    print(f"{len(df):,} filas · {len(df.columns)} columnas · guardado en «{nombre}»")
    display(df if len(df) <= 40 else df.head(20))

get_ipython().register_magic_function(_sql, "cell", "sql")
pd.options.display.float_format = lambda x: f"{x:,.2f}" if (abs(x) >= 1 or x == 0) else f"{x:.4f}"

def pregunta(texto, opciones):
    """Muestra una pregunta de predicción con sus opciones."""
    print(texto)
    for letra, opcion in zip("ABCD", opciones):
        print(f"   {letra}) {opcion}")

def comprobar(respuesta, correcta, explicacion):
    """Compara tu respuesta con la correcta y explica por qué."""
    if respuesta is None:
        print('👉 Primero escribí tu respuesta en la celda de arriba (por ejemplo: respuesta = "B") y volvé a correr esta celda.')
        return
    ok = str(respuesta).strip().upper() == correcta
    print(("✅ Correcto. " if ok else f"❌ Todavía no: la respuesta es la {correcta}. ") + explicacion)

def pistas(*textos):
    """Muestra pistas numeradas, de a una por línea."""
    for i, t in enumerate(textos, 1):
        print(f"Pista {i}: {t}")
print("pandas:", pd.__version__, "· duckdb:", duckdb.__version__)

## Paso 1 · Armamos los datos de Mercado Sur

No cargamos ningún archivo: **los generamos con código**, así todos tenemos exactamente los mismos datos. Son **tres tablas relacionadas**:

```
clientes ──(id_cliente)──▶ ventas ◀──(id_producto)── productos
quién compra                qué ocurrió               qué se vende
```

| Tabla | Una fila representa… |
| --- | --- |
| `clientes` | un cliente |
| `productos` | un producto del catálogo |
| `ventas` | **una operación de venta** (completada, devuelta o cancelada) |

### 1.1 · La semilla y los parámetros

Casi todo lo que sigue es **un sorteo** (precios, fechas, canales…). Para que el sorteo sea siempre el mismo usamos una **semilla**: con la misma semilla, cualquiera que corra este código obtiene exactamente los mismos datos.

In [ ]:
SEMILLA = 2025                                 # el «número de sorteo»
rng = np.random.default_rng(SEMILLA)           # el generador de sorteos
N_CLIENTES = 1500
N_OPS = 12000                                  # operaciones que se sortean (algunas se descartan más adelante)
print("Listo: vamos a generar", N_CLIENTES, "clientes y", N_OPS, "operaciones")

### 1.2 · La tabla `productos`

Cinco categorías, **ocho productos por categoría**. Cada categoría tiene un rango de precios (la electrónica es cara; los alimentos, baratos), y de ahí sale el `precio_lista` de cada producto.

In [ ]:
CATEGORIAS = {"Electrónica": (150, 900, 0.15), "Hogar": (30, 300, 0.20), "Indumentaria": (20, 150, 0.20),
              "Deportes": (25, 250, 0.15), "Alimentos": (3, 40, 0.30)}   # (precio mínimo, precio máximo, peso en las ventas)
filas, n = [], 1
for cat, (lo, hi, _) in CATEGORIAS.items():
    for _ in range(8):
        filas.append({"id_producto": f"P{n:03d}", "categoria": cat, "nombre": f"{cat} {n:02d}",
                      "precio_lista": round(float(rng.uniform(lo, hi)), 2)})
        n += 1
productos = pd.DataFrame(filas)
productos.head()

### 1.3 · La tabla `clientes`

Cada cliente tiene una **región** (Centro, Norte, Sur, Litoral o Cuyo) y una **fecha de alta**.

In [ ]:
REGIONES = ["Centro", "Norte", "Sur", "Litoral", "Cuyo"]
clientes = pd.DataFrame({
    "id_cliente": [f"C{i:04d}" for i in range(1, N_CLIENTES + 1)],
    "region": rng.choice(REGIONES, size=N_CLIENTES, p=[0.30, 0.20, 0.18, 0.17, 0.15]),
    "fecha_alta": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, N_CLIENTES), unit="D"),
})
clientes["fecha_alta"] = clientes["fecha_alta"].dt.strftime("%Y-%m-%d")
clientes.head()

### 1.4 · La tabla `ventas`: quién compró qué, cuándo y por qué canal

Ahora las **operaciones**. Para cada una sorteamos la **fecha** (durante 2025), **quién** compró, por qué **canal** (web o tienda) y **qué producto**. Después, las **unidades**, el **precio unitario** (cerca del precio de lista) y el **descuento**.

Para armarla usamos columnas auxiliares (`categoria`, `precio_lista`, `region`) que al final descartamos.

In [ ]:
fechas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, N_OPS), unit="D")
ventas = pd.DataFrame({
    "id_venta": [f"V{i:06d}" for i in range(1, N_OPS + 1)],
    "fecha": fechas,
    "id_cliente": rng.choice(clientes["id_cliente"], size=N_OPS),
    "canal": rng.choice(["web", "tienda"], size=N_OPS, p=[0.55, 0.45]),
})
pesos = np.array([v[2] for v in CATEGORIAS.values()])
cats = np.array(list(CATEGORIAS.keys()))[rng.choice(len(CATEGORIAS), size=N_OPS, p=pesos / pesos.sum())]
ventas["id_producto"] = [rng.choice(productos.loc[productos["categoria"] == c, "id_producto"]) for c in cats]
ventas = ventas.merge(productos[["id_producto", "categoria", "precio_lista"]], on="id_producto", how="left")
ventas = ventas.merge(clientes[["id_cliente", "region"]], on="id_cliente", how="left")
ventas["unidades"] = np.where(ventas["categoria"] == "Alimentos", rng.integers(1, 9, N_OPS), rng.integers(1, 5, N_OPS))
ventas["precio_unitario"] = (ventas["precio_lista"] * rng.uniform(0.97, 1.05, N_OPS)).round(2)
ventas["descuento_pct"] = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], size=N_OPS, p=[0.45, 0.15, 0.20, 0.12, 0.08])
print(f"{len(ventas):,} operaciones sorteadas")

### 1.5 · El estado de cada operación

Cada operación termina en uno de tres **estados**: `completada`, `devuelta` o `cancelada`. Casi todas se completan. Se genera con una regla interna que **no vamos a mostrar**: descubrir qué pasa es parte del trabajo de análisis.

In [ ]:
s2 = ventas["fecha"].dt.month > 6
seg = (ventas["canal"] == "web") & (ventas["region"] == "Litoral") & (ventas["categoria"] == "Electrónica") & s2
seg_indum = (ventas["canal"] == "web") & (ventas["categoria"] == "Indumentaria") & s2
p_comp, p_dev, p_can = np.full(N_OPS, 0.88), np.full(N_OPS, 0.07), np.full(N_OPS, 0.05)
p_comp[seg], p_dev[seg], p_can[seg] = 0.64, 0.30, 0.06
p_comp[seg_indum], p_dev[seg_indum], p_can[seg_indum] = 0.81, 0.13, 0.06
u = rng.random(N_OPS)
ventas["estado"] = np.where(u < p_comp, "completada", np.where(u < p_comp + p_dev, "devuelta", "cancelada"))
ventas = ventas.loc[~(seg & (rng.random(N_OPS) < 0.38))].copy()
print(f"Estados asignados · operaciones: {len(ventas):,}")

### 1.6 · La satisfacción (con faltantes)

Algunos clientes califican la compra de **1 a 5**. Pero **no todos responden**: la satisfacción puede faltar, y falta más en las operaciones canceladas y devueltas. Los faltantes **se analizan, no se reemplazan**.

In [ ]:
n_v = len(ventas)
sat = rng.choice([1, 2, 3, 4, 5], size=n_v, p=[0.05, 0.08, 0.17, 0.35, 0.35]).astype(float)
dev = (ventas["estado"] == "devuelta").to_numpy()
can = (ventas["estado"] == "cancelada").to_numpy()
sat[dev] = rng.choice([1, 2, 3], size=dev.sum(), p=[0.5, 0.3, 0.2])               # las devueltas califican peor
sat[rng.random(n_v) < np.where(can, 0.80, np.where(dev, 0.25, 0.12))] = np.nan     # y los faltantes
ventas["satisfaccion"] = sat

ventas = ventas.sort_values("fecha").reset_index(drop=True)                          # ordenadas por fecha
ventas["id_venta"] = [f"V{i:06d}" for i in range(1, len(ventas) + 1)]
ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
ventas = ventas[["id_venta", "fecha", "id_cliente", "id_producto", "canal", "unidades",
                 "precio_unitario", "descuento_pct", "estado", "satisfaccion"]]       # descartamos las columnas auxiliares
ventas.head()

Esto es lo que significa cada columna de `ventas`:

In [ ]:
pd.DataFrame({
    "columna": ["id_venta", "fecha", "id_cliente", "id_producto", "canal", "unidades", "precio_unitario",
                "descuento_pct", "estado", "satisfaccion"],
    "qué es": ["Identificador de la operación", "Día de la operación (2025)", "Quién compró (une con `clientes`)",
               "Qué se vendió (une con `productos`)", "web o tienda", "Cuántas unidades",
               "Precio por unidad en esa operación", "Descuento, como fracción de 0 a 1 (0.10 = 10 %)",
               "completada, devuelta o cancelada", "Encuesta de 1 a 5 (puede faltar)"]})

Por último, le contamos a DuckDB cuáles son las tres tablas, para poder consultarlas con SQL (las fechas pasan a ser fechas de verdad):

In [ ]:
con.register("clientes", clientes.assign(fecha_alta=pd.to_datetime(clientes["fecha_alta"])))
con.register("productos", productos)
con.register("ventas", ventas.assign(fecha=pd.to_datetime(ventas["fecha"])))
print({t: con.sql(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("clientes", "productos", "ventas")})

**Listo.** Quedaron registradas `clientes`, `productos` y `ventas`.

## Paso 2 · El problema: tres reportes, tres números

En Mercado Sur hay **tres reportes** que muestran el **ticket promedio** de un cliente (el monto neto promedio de sus pedidos). Para el cliente `C1042`, los tres muestran un número distinto.

| Reporte | Cómo calcula el ticket promedio |
| --- | --- |
| Marketing | Sobre **todos** los pedidos |
| Finanzas | Solo sobre los pedidos **entregados** |
| Operaciones | Sobre los entregados de los **últimos 12 meses** |

**Dos aclaraciones sobre los datos de práctica.** En Mercado Sur, un pedido «entregado» es el que tiene estado `completada`. Y «los últimos 12 meses» se cuenta hacia atrás desde una **fecha de corte**, que acá es el **30 de septiembre de 2025**.

In [ ]:
%%sql reportes
SELECT 'Marketing' AS reporte, 'todos los pedidos' AS regla, COUNT(*) AS pedidos,
       ROUND(AVG(unidades * precio_unitario * (1 - descuento_pct)), 2) AS ticket_promedio
  FROM ventas
 WHERE id_cliente = 'C1042'
UNION ALL
SELECT 'Finanzas', 'solo entregados (completada)', COUNT(*),
       ROUND(AVG(unidades * precio_unitario * (1 - descuento_pct)), 2)
  FROM ventas
 WHERE id_cliente = 'C1042' AND estado = 'completada'
UNION ALL
SELECT 'Operaciones', 'entregados de los últimos 12 meses', COUNT(*),
       ROUND(AVG(unidades * precio_unitario * (1 - descuento_pct)), 2)
  FROM ventas
 WHERE id_cliente = 'C1042' AND estado = 'completada'
   AND fecha > DATE '2025-09-30' - INTERVAL 12 MONTH AND fecha <= DATE '2025-09-30'

In [ ]:
pregunta("PREDICCIÓN 1 · Tres reportes dan tres números para el mismo cliente. ¿Cuál es el incorrecto?",
         ["El de Marketing", "El de Finanzas", "El de Operaciones", "Ninguno: cada uno usa una regla distinta"])
respuesta = None        # 👉 TU TURNO: cambiá None por tu letra, por ejemplo "D"

In [ ]:
comprobar(respuesta, "D", "Ninguna consulta está mal: cada una hace lo que dice. El problema es que la regla de cálculo vive copiada en tres lugares.")

**Qué vimos.** Ninguno está «mal» desde el punto de vista del SQL. El problema es **organizacional**: la regla vive copiada y pegada en tres lugares, y cuando alguien la corrige, corrige uno solo. Una **función** encapsula la regla, la nombra, la versiona y la deja disponible para cualquier consulta. Y queda una pregunta de fondo, la del consejo de la clase 1: **¿quién es el dueño de la definición?**

## Paso 3 · Una base de PL/SQL, en simple

Seis ideas, en este orden. Alcanzan para todo lo que sigue.

| # | Idea | En una frase |
| --- | --- | --- |
| 1 | SQL y PL/SQL | SQL describe un resultado; PL/SQL le agrega **pasos**: variables, condiciones y manejo de errores. Una función es, en el fondo, una **fórmula con nombre** |
| 2 | El bloque | Un bloque tiene `DECLARE` (opcional), `BEGIN` y `END`, y una sección `EXCEPTION` (opcional) para los errores |
| 3 | Variables y tipos | Se declara una variable con un nombre y un tipo (`v_ticket NUMBER`), o con el tipo de una columna con `%TYPE` |
| 4 | Función o procedimiento | Una **función devuelve un valor** y se usa dentro de un `SELECT`; un **procedimiento hace algo** (inserta, actualiza) y se invoca. Si querés un número para usar en una consulta, es una función |
| 5 | Parámetros | Lo que entra: el cliente, si cuenta solo entregados. Se declaran con su tipo y se les puede dar un valor por defecto |
| 6 | Qué pasa si no hay filas | Un promedio de cero pedidos devuelve `NULL`: no es un error, pero hay que decidir qué hacer con él y dejarlo escrito |

**La función de la clase, en PL/SQL** (especificada, no ejecutada acá):

```sql
CREATE OR REPLACE FUNCTION ticket_promedio (
    p_id_cliente       IN ventas.id_cliente%TYPE,
    p_solo_entregados  IN NUMBER DEFAULT 1
) RETURN NUMBER IS
    v_ticket NUMBER;
BEGIN
    SELECT AVG(unidades * precio_unitario * (1 - descuento_pct))
      INTO v_ticket
      FROM ventas
     WHERE id_cliente = p_id_cliente
       AND (p_solo_entregados = 0 OR estado = 'completada');
    RETURN v_ticket;
END ticket_promedio;
/
```

**Su equivalente ejecutable en DuckDB** es una *macro*:

| En Oracle | En DuckDB |
| --- | --- |
| `CREATE OR REPLACE FUNCTION` | `CREATE OR REPLACE MACRO` |
| `p_solo_entregados IN NUMBER DEFAULT 1` | `p_solo_entregados := 1` |
| `RETURN` de la variable | El resultado de la expresión |
| El catálogo: `USER_OBJECTS` y `USER_ERRORS` | El catálogo: `duckdb_functions()` |

## Paso 4 · Los siete pasos para desarrollar una función

Una pieza de PL/SQL no se escribe y se da por buena: se **desarrolla en pasos**, y cada uno deja una evidencia.

| # | Paso | En Oracle | En este notebook |
| --- | --- | --- | --- |
| 1 | Escribí en una línea qué hace y qué entra y sale | — | Una línea |
| 2 | Creá el objeto y cerrá con `/` | `CREATE OR REPLACE PROCEDURE` (o `FUNCTION`) | `CREATE OR REPLACE MACRO` |
| 3 | Confirmá que compiló. Si avisa «compilado con errores», pedí el detalle | `SHOW ERRORS` o la vista `USER_ERRORS` | El mensaje de error al crearla |
| 4 | Consultá el catálogo: que exista y esté válido | `USER_OBJECTS` y su columna de estado | `duckdb_functions()` |
| 5 | Invocalo con valores conocidos | `SELECT ticket_promedio('C1042') FROM dual` | `SELECT ticket_promedio('C1042')` |
| 6 | Validá con un `SELECT` independiente, no con la salida del propio objeto | Un `AVG` escrito a mano | Igual |
| 7 | Probá al menos un caso borde | Mes sin ventas, canal inexistente, mes 13 | Cliente inexistente, corte anterior a todo |

### Paso 1 de 7 · Una línea

> **`ticket_promedio` calcula el ticket promedio neto de un cliente. Entra: el cliente, si cuenta solo los entregados y, opcionalmente, cuántos meses mirar hacia atrás desde una fecha de corte. Sale: un número (o `NULL` si no hay pedidos).**

### Paso 2 de 7 · Crear el objeto

La regla se escribe **una sola vez**, adentro:

In [ ]:
con.sql("""
CREATE OR REPLACE MACRO ticket_promedio(p_id_cliente, p_solo_entregados := 1, p_meses := NULL, p_corte := DATE '2025-09-30') AS (
    SELECT AVG(unidades * precio_unitario * (1 - descuento_pct))              -- la regla, escrita una sola vez
      FROM ventas
     WHERE id_cliente = p_id_cliente                                         -- de este cliente
       AND (p_solo_entregados = 0 OR estado = 'completada')                  -- todos los pedidos o solo los entregados
       AND (p_meses IS NULL OR (fecha > p_corte - to_months(p_meses)         -- sin límite, o últimos N meses
                                AND fecha <= p_corte))
)""")
print("✅ Creada: ticket_promedio")

### Paso 3 de 7 · Confirmar que compiló

En Oracle, un objeto con un error se **crea igual pero queda inválido**, y el cliente avisa «compilado con errores»: hay que pedir el detalle. DuckDB es más estricto: **no la crea** y dice por qué. Probémoslo con una macro con un error de tipeo:

In [ ]:
try:
    con.sql("CREATE OR REPLACE MACRO macro_con_error(x) AS (SELECT AVG(columna_que_no_existe) FROM ventas WHERE id_cliente = x)")
except Exception as e:
    print("❌ No se creó. El detalle del error:")
    print("  ", str(e).split("\n")[0])

### Paso 4 de 7 · Consultar el catálogo

¿Existen las dos? Solo debería figurar la que se creó bien:

In [ ]:
%%sql catalogo
SELECT function_name, function_type, parameters
  FROM duckdb_functions()
 WHERE function_name IN ('ticket_promedio', 'macro_con_error')

### Paso 5 de 7 · Invocarla con valores conocidos

Sabemos qué números dieron los tres reportes. Si la función encapsula bien las tres reglas, debería devolver **los mismos tres números**, cambiando solo los parámetros.

In [ ]:
pregunta("PREDICCIÓN 2 · Llamamos a la función con los parámetros de cada reporte. ¿Qué esperás?",
         ["Tres números iguales", "Los mismos tres números de los reportes (521,66 · 538,79 · 617,08)", "Tres números nuevos, distintos de los reportes"])
respuesta = None        # 👉 TU TURNO

In [ ]:
comprobar(respuesta, "B", "La función no cambia las reglas: las guarda. Cada reporte pasa a ser una llamada con otros parámetros.")

In [ ]:
%%sql invocacion
SELECT ticket_promedio('C1042', 0)      AS marketing,
       ticket_promedio('C1042')         AS finanzas,
       ticket_promedio('C1042', 1, 12)  AS operaciones

### Paso 6 de 7 · Validar con un `SELECT` independiente

La función **no puede validarse a sí misma**: si tuviera un error de lógica, se confirmaría con el mismo error. Se compara contra una consulta escrita a mano:

In [ ]:
independiente = con.sql("""SELECT AVG(unidades * precio_unitario * (1 - descuento_pct))
                             FROM ventas WHERE id_cliente = 'C1042' AND estado = 'completada'""").fetchone()[0]
desde_la_funcion = con.sql("SELECT ticket_promedio('C1042')").fetchone()[0]
print(f"SELECT independiente: {independiente:,.4f}")
print(f"La función:           {desde_la_funcion:,.4f}")
print("✅ Coinciden" if abs(independiente - desde_la_funcion) < 1e-9 else "❌ No coinciden: hay un error de lógica")

### Paso 7 de 7 · Casos borde

Un caso feliz no alcanza: el error aparece con el primer dato raro. Cada prueba tiene una **entrada**, una **salida esperada** y la **salida obtenida**:

In [ ]:
pruebas = [
    ("Cliente que no existe",                       "SELECT ticket_promedio('C9999')",                         None),
    ("Corte anterior a todos los pedidos",          "SELECT ticket_promedio('C1042', 1, 12, DATE '2024-01-01')", None),
    ("Todos los pedidos (regla de Marketing)",      "SELECT ticket_promedio('C1042', 0)",
     con.sql("SELECT AVG(unidades * precio_unitario * (1 - descuento_pct)) FROM ventas WHERE id_cliente = 'C1042'").fetchone()[0]),
]
filas = []
for nombre, sql, esperado in pruebas:
    obtenido = con.sql(sql).fetchone()[0]
    ok = (obtenido is None and esperado is None) or (obtenido is not None and esperado is not None and abs(obtenido - esperado) < 1e-9)
    filas.append((nombre, "NULL" if esperado is None else f"{esperado:,.2f}", "NULL" if obtenido is None else f"{obtenido:,.2f}", "✅" if ok else "❌"))
pd.DataFrame(filas, columns=["caso", "esperado", "obtenido", "resultado"])

**Qué vimos.** Siete pasos, siete evidencias: la línea que dice qué hace, la creación, la compilación, el catálogo, una invocación con valores conocidos, una validación independiente y los casos borde. Ahora, las **tres reglas viven en un solo lugar**: si cambia la definición del ticket promedio, se corrige una vez.

## Paso 5 · Tu turno: una segunda función

Escribí la función `facturacion_mes_canal`, siguiendo los siete pasos. **Contrato:**

| | |
| --- | --- |
| **Entra** | El año, el mes y el canal |
| **Sale** | La facturación **neta** (unidades × precio × (1 − descuento)) de las operaciones `completada` de ese mes y canal |
| **Mes sin ventas** | Devuelve `0` |
| **Mes fuera de 1 a 12** | Devuelve `NULL` |
| **Canal que no existe** | Devuelve `NULL` |

El esqueleto está creado y **devuelve siempre `NULL`**: reemplazalo por tu regla. Cuando creas que está lista, corré el verificador de más abajo.

In [ ]:
con.sql("""
CREATE OR REPLACE MACRO facturacion_mes_canal(p_anio, p_mes, p_canal) AS (
    -- 👉 TU TURNO: escribí acá la regla (pistas en la celda siguiente)
    SELECT NULL
)""")
print("Esqueleto creado: facturacion_mes_canal (todavía devuelve NULL)")

In [ ]:
pistas("Primero decidí cuándo la entrada es válida: ¿el mes está entre 1 y 12? ¿el canal figura en la tabla ventas?",
       "Si la entrada no es válida, el resultado es NULL; en SQL eso lo expresa un CASE WHEN ... THEN ... END sin ELSE.",
       "Si la entrada es válida pero no hay ventas, SUM devuelve NULL: COALESCE(..., 0) lo convierte en 0.",
       "Para filtrar por año y mes podés usar year(fecha) y month(fecha).")

In [ ]:
def verificar_tu_turno():
    """Corre las pruebas del contrato y muestra cuáles pasan."""
    def esperado_a_mano(anio, mes, canal):
        return con.sql(f"""SELECT SUM(unidades * precio_unitario * (1 - descuento_pct)) FROM ventas
                            WHERE estado = 'completada' AND year(fecha) = {anio} AND month(fecha) = {mes} AND canal = '{canal}'""").fetchone()[0]
    casos = [("Mes con ventas (marzo, web)", (2025, 3, "web"), esperado_a_mano(2025, 3, "web")),
             ("Mes sin ventas (marzo de 2024)", (2024, 3, "web"), 0),
             ("Mes fuera de 1 a 12 (el 13)", (2025, 13, "web"), None),
             ("Canal que no existe", (2025, 3, "fax"), None)]
    filas, aprobadas = [], 0
    for nombre, (a, m, ca), esperado in casos:
        try:
            obtenido = con.sql(f"SELECT facturacion_mes_canal({a}, {m}, '{ca}')").fetchone()[0]
        except Exception as e:
            obtenido = f"error: {str(e).splitlines()[0][:50]}"
        ok = (obtenido is None and esperado is None) or (isinstance(obtenido, (int, float)) and esperado is not None and abs(obtenido - esperado) < 1e-6)
        aprobadas += ok
        filas.append((nombre, "NULL" if esperado is None else f"{esperado:,.2f}", "NULL" if obtenido is None else (f"{obtenido:,.2f}" if isinstance(obtenido, (int, float)) else obtenido), "✅" if ok else "❌"))
    display(pd.DataFrame(filas, columns=["caso", "esperado", "obtenido", "resultado"]))
    print(f"{aprobadas} de {len(casos)} pruebas aprobadas" + (" 🎉" if aprobadas == len(casos) else " · seguí iterando"))

verificar_tu_turno()

**Cuando termines:** pasale tu función a otro equipo. **Quien revisa no corrige**: escribe una prueba capaz de revelar un error de tu función.

## Paso 6 · Un error que compila y miente

Que un objeto **compile no significa que sea correcto**. Esta versión de `ticket_promedio` se crea sin ningún aviso:

In [ ]:
con.sql("""
CREATE OR REPLACE MACRO ticket_promedio_mal(p_id_cliente) AS (
    SELECT SUM(CASE WHEN estado = 'completada' THEN unidades * precio_unitario * (1 - descuento_pct) END) / COUNT(*)
      FROM ventas WHERE id_cliente = p_id_cliente
)""")
print("✅ Creada sin errores: ticket_promedio_mal")

In [ ]:
pregunta("PREDICCIÓN 3 · Para el cliente C1042, ¿qué número esperás de ticket_promedio_mal frente al de Finanzas (538,79)?",
         ["El mismo", "Menor", "Mayor"])
respuesta = None        # 👉 TU TURNO

In [ ]:
comprobar(respuesta, "B", "Suma solo lo entregado pero divide por todos los pedidos (incluida la cancelada), así que el promedio da menos.")

In [ ]:
mal = con.sql("SELECT ticket_promedio_mal('C1042')").fetchone()[0]
bien = con.sql("SELECT AVG(unidades * precio_unitario * (1 - descuento_pct)) FROM ventas WHERE id_cliente = 'C1042' AND estado = 'completada'").fetchone()[0]
print(f"ticket_promedio_mal:  {mal:,.2f}")
print(f"SELECT independiente: {bien:,.2f}")
print("❌ La validación independiente lo detectó: compila, pero miente")

**Qué vimos.** Este error lo atrapó el **paso 6**: un `SELECT` independiente. Por eso el objeto no puede validarse con su propia salida.

## Paso 7 · Optimizar una consulta con evidencia

Optimizar es un **método**, no una intuición. Este es el flujo:

1. **Medir:** cuánto tarda y con qué volumen.
2. **Pedir el plan** (`EXPLAIN PLAN` en Oracle; `EXPLAIN` en DuckDB).
3. **Ubicar la operación más costosa.**
4. **Decidir según el filtro:** ¿devuelve pocas filas?
   - **Sí →** destrabar predicados y evaluar un índice.
   - **No →** reducir columnas y filtrar antes de agregar.
5. **Cambiar una sola cosa.**
6. **Volver a pedir el plan y comparar el resultado.**

Para que haya algo que optimizar, agrandamos `ventas`: la repetimos 100 veces (con identificadores distintos) y obtenemos más de un millón de filas.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE ventas_grande AS
SELECT id_venta || '-' || r.range::VARCHAR AS id_venta, v.* EXCLUDE (id_venta)
  FROM ventas v, range(100) r
""")
print({t: con.sql(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("ventas", "ventas_grande")})

Y las **herramientas del flujo**: medir una consulta, ver su plan y comparar dos versiones comprobando que den el **mismo resultado**.

In [ ]:
def medir(sql, repeticiones=3):
    """Corre la consulta varias veces y se queda con el mejor tiempo (en segundos)."""
    mejor, resultado = None, None
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        resultado = con.sql(sql).df()
        seg = time.perf_counter() - t0
        mejor = seg if mejor is None else min(mejor, seg)
    return mejor, resultado

def plan(sql, analizar=False):
    """Muestra el plan de ejecución. Con analizar=True, además lo ejecuta y muestra el tiempo de cada operación."""
    filas = con.sql(("EXPLAIN ANALYZE " if analizar else "EXPLAIN ") + sql).fetchall()
    print(filas[0][1])

def mismo_resultado(a, b):
    a = a.sort_values(list(a.columns)).reset_index(drop=True)
    b = b.sort_values(list(b.columns)).reset_index(drop=True)
    try:
        pd.testing.assert_frame_equal(a, b, check_exact=False, rtol=1e-9, check_dtype=False)
        return True
    except AssertionError:
        return False

def comparar(sql_a, sql_b, repeticiones=3):
    """Mide las dos versiones y comprueba que devuelvan lo mismo."""
    ta, ra = medir(sql_a, repeticiones)
    tb, rb = medir(sql_b, repeticiones)
    rb.columns = ra.columns
    display(pd.DataFrame({"versión": ["A · inicial", "B · propuesta"], "segundos": [ta, tb], "filas del resultado": [len(ra), len(rb)]}).round({"segundos": 4}))
    razon = ta / tb
    print(f"B es {razon:,.1f} veces más rápida que A" if razon >= 1.3 else
          f"B es más lenta que A ({razon:.2f} veces)" if razon <= 0.77 else
          f"Sin diferencia relevante entre A y B (≈ {razon:.1f} veces)")
    print("✅ Mismo resultado" if mismo_resultado(ra, rb) else "❌ El resultado cambió: esta optimización no es válida")
print("Herramientas listas: medir · plan · comparar")

### Paso 1 del flujo · Medir: el volumen

In [ ]:
%%sql volumen
SELECT 'ventas_grande' AS tabla, COUNT(*) AS filas FROM ventas_grande
UNION ALL SELECT 'productos', COUNT(*) FROM productos
UNION ALL SELECT 'clientes', COUNT(*) FROM clientes

### Ejemplo 1 · El filtro devuelve pocas filas: destrabar predicados

**Pedido:** la facturación neta de **marzo de 2025**. Marzo es una fracción chica del año: el filtro **devuelve pocas filas**, así que vamos por la rama **Sí**: destrabar el predicado.

- **A (inicial):** convierte la fecha a texto (`strftime`) para comparar el mes.
- **B (propuesta):** compara la fecha **directamente**, con un rango.

In [ ]:
consulta_a = """SELECT SUM(unidades * precio_unitario * (1 - descuento_pct)) AS facturacion
                  FROM ventas_grande
                 WHERE strftime(fecha, '%Y-%m') = '2025-03'"""
consulta_b = """SELECT SUM(unidades * precio_unitario * (1 - descuento_pct)) AS facturacion
                  FROM ventas_grande
                 WHERE fecha >= DATE '2025-03-01' AND fecha < DATE '2025-04-01'"""
pregunta("PREDICCIÓN 4 · Las dos consultas devuelven lo mismo. ¿Cuál esperás que sea más rápida?",
         ["La A, la de la función sobre la fecha", "La B, la del rango", "Las dos, igual"])
respuesta = None        # 👉 TU TURNO

In [ ]:
comprobar(respuesta, "B", "Con la función, el motor tiene que calcular un texto para cada fila antes de poder comparar; con el rango, compara fechas directamente.")

**Pedimos el plan** de cada versión. Buscá en el cuadro `SEQ_SCAN` la línea `Filters:`:

In [ ]:
print("──────── Plan de A ────────")
plan(consulta_a)
print("──────── Plan de B ────────")
plan(consulta_b)

Y **comparamos**: tiempo, filas y, sobre todo, que el resultado sea el mismo.

In [ ]:
comparar(consulta_a, consulta_b)

**Qué vimos.** El plan de A filtra con una **función sobre la columna**; el de B, con una **comparación directa**. Esa diferencia en la línea `Filters:` es la que se traduce en tiempo. **Un solo cambio, el mismo resultado.**

### Ejemplo 2 · Contraste: cuando el plan no cambia

Una intuición habitual: «si el filtro está en el `HAVING`, la consulta es más lenta que con `WHERE`». Probémosla, **sin suponerla**:

- **A:** `... GROUP BY canal HAVING canal = 'web'`
- **B:** `... WHERE canal = 'web' GROUP BY canal`

In [ ]:
consulta_a = """SELECT canal, SUM(unidades * precio_unitario * (1 - descuento_pct)) AS facturacion
                  FROM ventas_grande GROUP BY canal HAVING canal = 'web'"""
consulta_b = """SELECT canal, SUM(unidades * precio_unitario * (1 - descuento_pct)) AS facturacion
                  FROM ventas_grande WHERE canal = 'web' GROUP BY canal"""
comparar(consulta_a, consulta_b)

In [ ]:
print("──────── Plan de A ────────")
plan(consulta_a)

**Qué vimos.** Sin diferencia relevante: el motor **ya empuja el filtro** hacia la lectura de la tabla, y los dos planes son equivalentes. La intuición era errónea, y **solo el plan lo mostró**. Conclusión de método: **no se cambia una consulta que ya está bien resuelta por el motor**, y no se promete una mejora que no se midió.

> **Costo estimado no es tiempo garantizado.** Un plan es una *estimación* de cómo va a trabajar el motor; la medición es lo que confirma el efecto.

## Paso 8 · La plantilla de evidencia de seis campos

Cada pieza que entregues lleva **seis campos**. Este es un ejemplo completo con la función de la clase; la segunda fila es para tu equipo:

In [ ]:
plantilla = pd.DataFrame([
    {"entrada": "id del cliente, si cuenta solo entregados, meses hacia atrás",
     "salida": "un número (o NULL si no hay pedidos)",
     "regla": "promedio de unidades × precio × (1 − descuento) sobre los pedidos elegidos",
     "caso de prueba": "C1042 → 521,66 · 538,79 · 617,08 (los tres reportes)",
     "resultado": "coincide con el SELECT independiente",
     "costo observado": "una consulta sobre ventas; sin diferencia de plan que reportar"},
    {c_: "" for c_ in ("entrada", "salida", "regla", "caso de prueba", "resultado", "costo observado")},
], index=["ticket_promedio", "tu equipo"])
plantilla.T          # 👉 TU TURNO: completá la columna «tu equipo» para tu desafío

---
## Desafío por equipos

Cada equipo recibe **un pedido distinto**: una consulta que hay que mejorar **sin cambiar sus filas ni sus métricas**. Trabajan con el archivo **`datos_desafio_sql_clase2.ipynb`**, que contiene solo los datos, y entregan la **plantilla de seis campos**:

1. La **versión inicial** y la **propuesta**.
2. El **control de equivalencia** (y el supuesto que lo sostiene, si lo hay).
3. El **plan antes y después**, y qué cambió en él.
4. El **costo observado**, sin prometer lo que no se midió.

**Las cuatro preguntas del facilitador:** ¿qué resultado debe permanecer igual? ¿dónde está el costo? ¿qué cambio hicieron? ¿cómo probaron que no alteraron el significado?

## Cierre

**Tres frases para completar, por escrito:**

1. *La pieza reutilizable evita…*
2. *La prueba independiente protege…*
3. *La optimización con evidencia mejora…, pero no promete…*